In [ ]:
import os, shutil, datetime
# 1️⃣ Imports
# ===============================
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns


In [ ]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1️⃣ Paths & Local dirs
# =======================
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT80_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split80_20"  # train/ val/
TEST_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"  # test

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/Emotion_DenseNet121_80_20_Results_{ts}"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# =======================
# 2️⃣ Copy train/val/test locally
# =======================
def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

print("Copying Train/Val/Test to Colab local...")
fresh_copy(os.path.join(SPLIT80_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT80_DIR, "val"), LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copied all data.")


Copying Train/Val/Test to Colab local...
✅ Copied all data.


In [ ]:
# 2️⃣ Hyperparameters
# ===============================
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
NUM_CLASSES = 7
EPOCHS = 30
LEARNING_RATE = 1e-3


In [ ]:
# ===============================
# 3️⃣ Data Generators (no data augmentation as requested)
# ===============================
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    LOCAL_TRAIN,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    LOCAL_VAL,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    LOCAL_TEST,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 22965 images belonging to 7 classes.
Found 5744 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [ ]:
# ===============================
# 4️⃣ Build DenseNet121 Model (Fine-tune)
# ===============================
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze base layers first
for layer in base_model.layers:
    layer.trainable = False

# Compile model
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
# ===============================
# 5️⃣ Callbacks
# ===============================
checkpoint = ModelCheckpoint(
    os.path.join(OUT_DIR, "best_model.h5"),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

callbacks = [checkpoint, reduce_lr, early_stop]


In [ ]:
# ===============================
# 6️⃣ Train Stage 1 (Freeze Base)
# ===============================
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.2703 - loss: 1.9812

718/718 ━━━━━━━━━━━━━━━━━━━━ 142s 158ms/step - accuracy: 0.2704 - loss: 1.9809 - val_accuracy: 0.4326 - val_loss: 1.5027 - learning_rate: 0.0010
Epoch 2/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.3818 - loss: 1.6192

718/718 ━━━━━━━━━━━━━━━━━━━━ 73s 101ms/step - accuracy: 0.3818 - loss: 1.6191 - val_accuracy: 0.4528 - val_loss: 1.4801 - learning_rate: 0.0010
Epoch 3/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 77s 107ms/step - accuracy: 0.3830 - loss: 1.5952 - val_accuracy: 0.4288 - val_loss: 1.4964 - learning_rate: 0.0010
Epoch 4/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4031 - loss: 1.5733

718/718 ━━━━━━━━━━━━━━━━━━━━ 73s 102ms/step - accuracy: 0.4031 - loss: 1.5733 - val_accuracy: 0.4586 - val_loss: 1.4548 - learning_rate: 0.0010
Epoch 5/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 86s 108ms/step - accuracy: 0.3898 - loss: 1.5804 - val_accuracy: 0.4264 - val_loss: 1.4723 - learning_rate: 0.0010
Epoch 6/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3931 - loss: 1.5884

718/718 ━━━━━━━━━━━━━━━━━━━━ 73s 101ms/step - accuracy: 0.3931 - loss: 1.5884 - val_accuracy: 0.4596 - val_loss: 1.4486 - learning_rate: 0.0010
Epoch 7/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 86s 107ms/step - accuracy: 0.3961 - loss: 1.5745 - val_accuracy: 0.4558 - val_loss: 1.4685 - learning_rate: 0.0010
Epoch 8/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 70s 98ms/step - accuracy: 0.3995 - loss: 1.5792 - val_accuracy: 0.4297 - val_loss: 1.4845 - learning_rate: 0.0010
Epoch 9/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3970 - loss: 1.5752
Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
718/718 ━━━━━━━━━━━━━━━━━━━━ 70s 98ms/step - accuracy: 0.3970 - loss: 1.5752 - val_accuracy: 0.4434 - val_loss: 1.4780 - learning_rate: 0.0010
Epoch 10/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 82s 98ms/step - accuracy: 0.4065 - loss: 1.5486 - val_accuracy: 0.4593 - val_loss: 1.4486 - learning_rate: 5.0000e-04


In [ ]:
# ===============================
# 7️⃣ Fine-tune Stage 2 (Unfreeze top layers)
# ===============================
for layer in base_model.layers[-50:]:  # fine-tune last 50 layers
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.3947 - loss: 1.8346

718/718 ━━━━━━━━━━━━━━━━━━━━ 157s 165ms/step - accuracy: 0.3947 - loss: 1.8345 - val_accuracy: 0.5385 - val_loss: 1.2573 - learning_rate: 1.0000e-04
Epoch 2/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.4818 - loss: 1.4701

718/718 ━━━━━━━━━━━━━━━━━━━━ 78s 109ms/step - accuracy: 0.4818 - loss: 1.4701 - val_accuracy: 0.5648 - val_loss: 1.1878 - learning_rate: 1.0000e-04
Epoch 3/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.5226 - loss: 1.3210

718/718 ━━━━━━━━━━━━━━━━━━━━ 85s 118ms/step - accuracy: 0.5226 - loss: 1.3210 - val_accuracy: 0.5721 - val_loss: 1.1385 - learning_rate: 1.0000e-04
Epoch 4/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.5503 - loss: 1.2151

718/718 ━━━━━━━━━━━━━━━━━━━━ 78s 109ms/step - accuracy: 0.5503 - loss: 1.2151 - val_accuracy: 0.5900 - val_loss: 1.1049 - learning_rate: 1.0000e-04
Epoch 5/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.5969 - loss: 1.0712

718/718 ━━━━━━━━━━━━━━━━━━━━ 78s 108ms/step - accuracy: 0.5969 - loss: 1.0712 - val_accuracy: 0.5951 - val_loss: 1.0996 - learning_rate: 1.0000e-04
Epoch 6/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.6434 - loss: 0.9607

718/718 ━━━━━━━━━━━━━━━━━━━━ 85s 118ms/step - accuracy: 0.6434 - loss: 0.9607 - val_accuracy: 0.6045 - val_loss: 1.0758 - learning_rate: 1.0000e-04
Epoch 7/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6826 - loss: 0.8583

718/718 ━━━━━━━━━━━━━━━━━━━━ 78s 109ms/step - accuracy: 0.6826 - loss: 0.8583 - val_accuracy: 0.6107 - val_loss: 1.0827 - learning_rate: 1.0000e-04
Epoch 8/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7327 - loss: 0.7286

718/718 ━━━━━━━━━━━━━━━━━━━━ 81s 108ms/step - accuracy: 0.7327 - loss: 0.7286 - val_accuracy: 0.6233 - val_loss: 1.1013 - learning_rate: 1.0000e-04
Epoch 9/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7731 - loss: 0.6181
Epoch 9: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
718/718 ━━━━━━━━━━━━━━━━━━━━ 75s 105ms/step - accuracy: 0.7731 - loss: 0.6181 - val_accuracy: 0.6128 - val_loss: 1.1559 - learning_rate: 1.0000e-04
Epoch 10/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 75s 104ms/step - accuracy: 0.8236 - loss: 0.4924 - val_accuracy: 0.6201 - val_loss: 1.1670 - learning_rate: 5.0000e-05
Epoch 11/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 83s 105ms/step - accuracy: 0.8556 - loss: 0.4160 - val_accuracy: 0.6139 - val_loss: 1.2193 - learning_rate: 5.0000e-05
Epoch 12/20
718/718 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.8792 - loss: 0.3563
Epoch 12: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
718/718 ━━━━━━━━━━━━━━━━━━━━ 82s 114ms/step - accuracy: 

In [ ]:
# ===============================
# 8️⃣ Evaluation on Test Set
# ===============================
test_steps = math.ceil(test_generator.samples / BATCH_SIZE)
test_loss, test_acc = model.evaluate(test_generator, steps=test_steps)
print(f"Test Accuracy: {test_acc:.4f}")


225/225 ━━━━━━━━━━━━━━━━━━━━ 30s 134ms/step - accuracy: 0.5252 - loss: 1.2424
Test Accuracy: 0.5947


In [ ]:
# Predictions
y_true = test_generator.classes
y_pred_prob = model.predict(test_generator, steps=test_steps)
y_pred = np.argmax(y_pred_prob, axis=1)
class_labels = list(test_generator.class_indices.keys())

# Classification Report
report = classification_report(y_true, y_pred, target_names=class_labels)
print(report)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_labels, yticklabels=class_labels, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"))
plt.close()

# ROC Curves (One vs Rest)
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
plt.figure(figsize=(12,10))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_labels[i]} (AUC = {roc_auc:.2f})")
plt.plot([0,1], [0,1], 'k--')
plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"))
plt.close()

# Accuracy & Loss Curves
def plot_history(history, stage):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc)+1)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(epochs, acc, label='Train Acc')
    plt.plot(epochs, val_acc, label='Val Acc')
    plt.title(f"{stage} Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs, loss, label='Train Loss')
    plt.plot(epochs, val_loss, label='Val Loss')
    plt.title(f"{stage} Loss")
    plt.legend()

    plt.savefig(os.path.join(OUT_DIR, f"{stage}_acc_loss.png"))
    plt.close()

plot_history(history_stage1, "Stage1_Freeze")
plot_history(history_stage2, "Stage2_Finetune")

# Save final model
model.save(os.path.join(OUT_DIR, "DenseNet121_final_model.h5"))
print(f"✅ All results saved to {OUT_DIR}")

225/225 ━━━━━━━━━━━━━━━━━━━━ 47s 142ms/step
              precision    recall  f1-score   support

       angry       0.48      0.48      0.48       958
   disgusted       0.81      0.23      0.36       111
     fearful       0.47      0.33      0.38      1024
       happy       0.82      0.80      0.81      1774
     neutral       0.53      0.63      0.58      1233
         sad       0.44      0.52      0.48      1247
   surprised       0.74      0.72      0.73       831

    accuracy                           0.59      7178
   macro avg       0.61      0.53      0.55      7178
weighted avg       0.60      0.59      0.59      7178



✅ All results saved to /content/drive/MyDrive/Colab Notebooks/Emotion_DenseNet121_80_20_Results_20250915_141322
